# Frozen candidate CPU validation analysis

This notebook analyzes the 30 reused and 20 new validation-only runs. It contains no training commands and does not load held-out test data.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
ROOT=Path('/content/drive/MyDrive/VitalDB-Feature-Selection')
REPO_URL='https://github.com/khyitty/VitalDB-Feature-Selection.git'
REPO_DIR=Path('/content/VitalDB-Feature-Selection')
DATASET_DIR=ROOT/'data/modeling/full'
SELECTION_DIR=ROOT/'outputs/predictive_feature_selection_30s'
RETRAINING_ROOT=ROOT/'outputs/frozen_candidate_retraining_validation_only'
REGISTRY=RETRAINING_ROOT/'candidate_source_registry.json'
OUTPUT_DIR=RETRAINING_ROOT/'analysis'

In [ ]:
import os, subprocess, sys
if (REPO_DIR/'.git').is_dir(): subprocess.run(['git','-C',str(REPO_DIR),'pull','--ff-only'],check=True)
else: subprocess.run(['git','clone',REPO_URL,str(REPO_DIR)],check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable,'scripts/install_colab_dependencies.py','--requirements','requirements-colab.txt'],check=True)
print('Commit:',subprocess.run(['git','rev-parse','HEAD'],check=True,capture_output=True,text=True).stdout.strip())

In [ ]:
subprocess.run([sys.executable,'scripts/analyze_frozen_candidates.py','--registry',str(REGISTRY),'--candidate-subsets',str(SELECTION_DIR/'candidate_subsets.json'),'--dataset-dir',str(DATASET_DIR),'--output-dir',str(OUTPUT_DIR),'--bootstrap-replicates','10000','--bootstrap-seed','20260717'],check=True)

In [ ]:
import pandas as pd
from IPython.display import Image, display
for name in ('candidate_validation_aggregate.csv','paired_candidate_contrasts.csv','paired_model_contrasts.csv','hierarchical_bootstrap_candidate_contrasts.csv','hierarchical_bootstrap_model_contrasts.csv','candidate_pareto.csv','candidate_evidence_table.csv'):
    print(name); display(pd.read_csv(OUTPUT_DIR/name))
for path in sorted((OUTPUT_DIR/'figures').glob('*.png')): print(path.name); display(Image(filename=str(path)))
print((OUTPUT_DIR/'frozen_candidate_validation_report.md').read_text(encoding='utf-8')[:15000])